In [1]:
import numpy as np
import h5py
import os 
import torch

In [11]:
data_path="/global/cfs/cdirs/m4259/ecucuzzella/soma_ppe_data/ml_converted/month_1/thedataset-redi-2.hdf5"
with h5py.File(data_path, 'r') as f:
    keys = list(f.keys())
    print(keys)
    print(len(keys))
    array=f[keys[0]][:]
    print(array.shape)

['forward_0', 'forward_1', 'forward_10', 'forward_11', 'forward_12', 'forward_13', 'forward_14', 'forward_15', 'forward_16', 'forward_17', 'forward_18', 'forward_19', 'forward_2', 'forward_20', 'forward_21', 'forward_22', 'forward_23', 'forward_24', 'forward_25', 'forward_26', 'forward_27', 'forward_28', 'forward_29', 'forward_3', 'forward_30', 'forward_31', 'forward_32', 'forward_33', 'forward_34', 'forward_35', 'forward_36', 'forward_37', 'forward_38', 'forward_39', 'forward_4', 'forward_40', 'forward_41', 'forward_42', 'forward_43', 'forward_44', 'forward_45', 'forward_46', 'forward_47', 'forward_48', 'forward_49', 'forward_5', 'forward_50', 'forward_51', 'forward_52', 'forward_53', 'forward_54', 'forward_55', 'forward_56', 'forward_57', 'forward_58', 'forward_59', 'forward_6', 'forward_60', 'forward_61', 'forward_62', 'forward_63', 'forward_64', 'forward_65', 'forward_66', 'forward_67', 'forward_68', 'forward_69', 'forward_7', 'forward_70', 'forward_71', 'forward_72', 'forward_73',

In [3]:
# save data as np array

data_path="/global/cfs/cdirs/m4259/ecucuzzella/soma_ppe_data/ml_converted/month_1/thedataset-impliciBottomDrag.hdf5"
data_dir = '/pscratch/sd/g/gzhao27/INR/SOMA/results/impliciBottomDrag_np'

os.makedirs(data_dir, exist_ok=True)
# if not os.path.exists(data_dir):
#     os.makedirs(data_dir)




In [27]:
with h5py.File(data_path, 'r') as f:
    keys = list(f.keys())
    
    for key in keys:
        array = f[key][:]
        sample_path = os.path.join(data_dir, key+'.npy')
        np.save(sample_path, array)

In [32]:
def my_generator(directory):
    """
    Generator function to yield numpy arrays from .npy files in a directory.

    Args:
        directory (str): Path to the directory containing .npy files.

    Yields:
        ndarray: The contents of each .npy file.
    """
    # List and sort the .npy files
    files = sorted([f for f in os.listdir(directory) if f.endswith('.npy')])
    
    for file_name in files:
        file_path = os.path.join(directory, file_name)
        print(f"Loading: {file_path}")  # Optional logging
        data = np.load(file_path)
        yield data

In [33]:
from mmap_ninja import RaggedMmap
RaggedMmap.from_generator(out_dir="/pscratch/sd/g/gzhao27/INR/SOMA/results/soma_mmap_save", 
                                       sample_generator=my_generator(data_dir), 
                                       batch_size=2)

Loading: /pscratch/sd/g/gzhao27/INR/SOMA/results/impliciBottomDrag_np/forward_0.npy
Loading: /pscratch/sd/g/gzhao27/INR/SOMA/results/impliciBottomDrag_np/forward_1.npy
Loading: /pscratch/sd/g/gzhao27/INR/SOMA/results/impliciBottomDrag_np/forward_10.npy
Loading: /pscratch/sd/g/gzhao27/INR/SOMA/results/impliciBottomDrag_np/forward_11.npy
Loading: /pscratch/sd/g/gzhao27/INR/SOMA/results/impliciBottomDrag_np/forward_12.npy
Loading: /pscratch/sd/g/gzhao27/INR/SOMA/results/impliciBottomDrag_np/forward_13.npy
Loading: /pscratch/sd/g/gzhao27/INR/SOMA/results/impliciBottomDrag_np/forward_14.npy
Loading: /pscratch/sd/g/gzhao27/INR/SOMA/results/impliciBottomDrag_np/forward_15.npy
Loading: /pscratch/sd/g/gzhao27/INR/SOMA/results/impliciBottomDrag_np/forward_16.npy
Loading: /pscratch/sd/g/gzhao27/INR/SOMA/results/impliciBottomDrag_np/forward_17.npy
Loading: /pscratch/sd/g/gzhao27/INR/SOMA/results/impliciBottomDrag_np/forward_18.npy
Loading: /pscratch/sd/g/gzhao27/INR/SOMA/results/impliciBottomDrag_

<mmap_ninja.ragged.RaggedMmap object at 0x7fd55d5f1ca0> of length: 92

In [3]:
from mmap_ninja import RaggedMmap
images_mmap = RaggedMmap("/pscratch/sd/g/gzhao27/INR/SOMA/results/soma_mmap_save")

In [7]:
# test read time and toch numpy time...
hdf5_file = h5py.File(data_path, 'r')
from time import time
start = time()

for i in range(10):
    _data = hdf5_file[keys[0]][:]
    # _data = torch.from_numpy(_data)
print(time()-start)

11.049710035324097


In [10]:
from time import time
start = time()

for i in range(100):
    _data = torch.from_numpy(images_mmap[0])
    # _data = torch.from_numpy(_data)
print(time()-start)

0.004221677780151367
